# Environment

In [ ]:
import os
os.chdir("../..")

In [ ]:
import json
import re

import pandas as pd

from src.utils.path import PROCESSED_DATA_PATH, PROCESSED_LEXICONS_PATH, FEATURES_PATH

## Load EmoWords Lexicon

In [ ]:
with open(f"{PROCESSED_LEXICONS_PATH}/emo_words.json") as f:
    emo_words = json.load(f)

# Feature Extraction Pipeline

In [ ]:
polarity = {
    "alegria": 1,
    "surpresa": 1,
    "medo": -1,
    "tristeza": -1,
    "raiva": -1,
    "desgosto": -1
}

In [ ]:
lex_neg = {k: emo_words[k] for k in ["raiva", "desgosto"]}

In [ ]:
class PipelineEmo:
    def __init__(self, lexicon, polarity):
        self.lexicon = lexicon
        self.polarity = polarity

    def searchEmo(self, word):
        for k, v in self.lexicon.items():
            if word in v:
                return k
        return None

    def extractWords(self, text):
        text = re.sub(r"[^\w\s]", "", text.lower())
        words = text.split()
        emo_found, words_found = [], []
        for w in words:
            emo = self.searchEmo(w)
            if emo is not None:
                emo_found.append(emo)
                words_found.append(w)
        return emo_found, words_found

    def getFeatWords(self, df):
        df[["emotions", "words"]] = df.apply(
            lambda row: self.extractWords(row["text"]),
            axis=1, result_type="expand"
        )
        return df

    def getFeatFrequency(self, df):
        total_terms = df["text"].apply(lambda x: len(x.split()))
        total_words = df["words"].apply(lambda x: len(x))
        df["emow"] = total_words / total_terms
        return df

In [ ]:
pipeline = PipelineEmo(lex_neg, polarity)

# Feature Engineering in Corpora

## Ited-BR

In [ ]:
ited = pd.read_csv(f"{PROCESSED_DATA_PATH}/ited-br.csv")

In [ ]:
ited = pipeline.getFeatWords(ited)
ited = pipeline.getFeatFrequency(ited).drop(columns=["emotions", "words"])

In [ ]:
ited[["emow"]].to_csv(f"{FEATURES_PATH}/ited-br-emow.csv", index=False)

## Unified-Hate

In [ ]:
unified = pd.read_csv(f"{PROCESSED_DATA_PATH}/unified-hate.csv")

In [ ]:
unified = pipeline.getFeatWords(unified)
unified = pipeline.getFeatFrequency(unified).drop(columns=["emotions", "words"])

In [ ]:
unified[["emow"]].to_csv(f"{FEATURES_PATH}/unified-hate-emow.csv", index=False)

## Hate-BR

In [ ]:
hate = pd.read_csv(f"{PROCESSED_DATA_PATH}/hate-br.csv")

In [ ]:
hate = pipeline.getFeatWords(hate)
hate = pipeline.getFeatFrequency(hate).drop(columns=["emotions", "words"])

In [ ]:
hate[["emow"]].to_csv(f"{FEATURES_PATH}/hate-br-emow.csv", index=False)